# Task11 - [Advanced] Driver Consistency Score

This consistency score combines two factors — the variability of a driver's finishing positions and their reliability in finishing races — weighted 60% and 40% respectively.

**Standard deviation score** (lower variance → higher score):

$$
\text{std\_score} = \frac{1}{1 + \sigma_{\text{position}}}
$$

**DNF (reliability) score:**

$$
\text{dnf\_score} = 1 - \frac{\text{DNF count}}{\text{Total races}}
$$

**Final consistency score:**

$$
\text{Consistency Score} = 0.6 \times \text{std\_score} + 0.4 \times \text{dnf\_score}
$$

Because raw position variance alone made backmarkers who consistently finished near the back look more "consistent" than actual front-runners, the analysis was restricted to each season's **top 10 point scorers**, so the score measures stability among drivers who were already competitive rather than just steady mediocrity. Validating the results against real F1 history (Schumacher's 2002 season, Vettel's 2011 and 2013 seasons, Verstappen's 2023 season) showed the top scores aligned closely with widely recognized dominant seasons, confirming the metric behaves as intended.

In [ ]:
# Design your own formula/metric to measure a driver's "consistency" across a season (you decide what factors matter - e.g. variance in finishing position, DNF rate, points per race, etc)
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_f1.csv')

subset = df[['year', 'full_name', 'raceId', 'position', 'statusId', 'points']]

df_status = pd.read_csv('../data/status.csv')
subset = subset.merge(df_status, on='statusId', how='left')

finished_pattern = subset['status'].str.contains(r'^\+\d+ Lap', regex=True)
finished_exact = subset['status'] == 'Finished'
subset['is_finished'] = finished_pattern | finished_exact

result = subset.groupby(['year', 'full_name']).agg(
    avg_position=('position', 'mean'),
    std_position=('position', 'std'),
    race_count=('raceId', 'count'),
    dnf_count=('is_finished', lambda x: (~x).sum()),
    total_points=('points', 'sum')
).reset_index()

result['dnf_rate'] = result['dnf_count'] / result['race_count']
result = result[result['race_count'] >= 5]
result = result[result['total_points'] > 0]

result = result.sort_values(['year', 'total_points'], ascending=[True, False])
result = result.groupby('year').head(10)

result['std_score'] = 1 / (1 + result['std_position'])
result['dnf_score'] = 1 - result['dnf_rate']
result['consistency_score'] = (
    result['std_score'] * 0.6 +
    result['dnf_score'] * 0.4
)

result.to_csv('../data/consistency_scores.csv', index=False)

for year in range(2000, 2025):
    year_data = result[result['year'] == year]
    top5 = year_data.sort_values('consistency_score', ascending=False).head(5)
    print(f"=== {year}년 Top 5 ===")
    print(top5[['full_name', 'consistency_score', 'total_points']])
    print()

# Task12 - [Advanced] Constructor Championship Simulation

In [ ]:
# Using historical points-per-position rules, write a function that recalculates the final constructor standings for a given year if the points system were different (e.g. apply 2020 rules to 1990 season, or design your own points scale)
# Compare original standings vs simulated standings for at least 1 season
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_f1.csv')

f1_points_systems = {
    # 1950-1957: top 5 finishers, fastest lap bonus point existed separately
    (1950, 1957): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2},

    # 1958-1959: top 5 finishers, shared-points rule removed
    (1958, 1959): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2},

    # 1960: top 6 finishers, fastest lap bonus removed
    (1960, 1960): {1: 8, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 1961-1990: top 6 finishers, win worth 9 points (longest-running system)
    (1961, 1990): {1: 9, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 1991-2002: top 6 finishers, win worth 10 points
    (1991, 2002): {1: 10, 2: 6, 3: 4, 4: 3, 5: 2, 6: 1},

    # 2003-2009: expanded to top 8 finishers
    (2003, 2009): {1: 10, 2: 8, 3: 6, 4: 5, 5: 4, 6: 3, 7: 2, 8: 1},

    # 2010-2024: top 10 finishers, win worth 25 points (current era)
    (2010, 2024): {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1},
}

def get_points_system(year):
    # Look up which points system applies to a given year
    for (start, end), system in f1_points_systems.items():
        if start <= year <= end:
            return system
    raise ValueError(f"No points system found for year {year}")

def calculate_points(position, points_system):
    if pd.isna(position):
        return 0
    position = int(position)
    return points_system.get(position, 0)

def simulate_standings(year, points_system):
    # Filter data for the given year and copy to avoid warnings
    season_data = df[df['year'] == year].copy()
    
    # Apply the points system to each row's finishing position
    season_data['simulated_points'] = season_data['position'].apply(
        lambda x: calculate_points(x, points_system)
    )
    
    # Sum up the ORIGINAL points per constructor (real historical result)
    original_standings = season_data.groupby('constructorId')['points'].sum().reset_index()
    original_standings = original_standings.rename(columns={'points': 'original_points'})
    
    # Sum up the SIMULATED points per constructor (using the new system)
    simulated_standings = season_data.groupby('constructorId')['simulated_points'].sum().reset_index()
    
    # Merge both standings side by side on constructor name
    comparison = original_standings.merge(simulated_standings, on='constructorId', how='left')
    
    # Rank both columns (higher points = better rank, i.e. rank 1)
    comparison['original_rank'] = comparison['original_points'].rank(ascending=False, method='min').astype(int)
    comparison['simulated_rank'] = comparison['simulated_points'].rank(ascending=False, method='min').astype(int)
    
    # Sort by original rank for readability
    comparison = comparison.sort_values('original_rank')
    
    return comparison

# Ask the user which year they want to recalculate
year = int(input("Enter the year to recalculate: "))

# Ask which year's points system to apply
system_year = int(input("Which year's points system do you want to apply? (e.g. 2020): "))
points_system = get_points_system(system_year)   

result = simulate_standings(year, points_system)

# Output both tables side by side
constructors_lookup = df[['constructorId', 'name']].drop_duplicates()
result_named = result.merge(constructors_lookup, on='constructorId', how='left')
final_table = result_named[['name', 'original_points', 'original_rank', 'simulated_points', 'simulated_rank']]
final_table['rank_change'] = final_table['original_rank'] - final_table['simulated_rank']
final_table

# Task13 - Circuit Difficulty Analysis

# Circuit Difficulty / Unpredictability Analysis — Methodology Summary

## 1. Objective

The goal was to rank F1 circuits by how "difficult" or "unpredictable" they are,
using only the data available in the dataset: finishing status (DNF reasons),
grid vs. finish position, lap times, and pit stops.

Early on, we distinguished two related but different concepts:
- **Difficulty**: how physically challenging a circuit is to drive
- **Unpredictability**: how much race results deviate from expectations

Given the available data (results, status, lap times, pit stops), the metric we
built turned out to measure **unpredictability** more directly than physical
difficulty — this becomes clear later in the findings.

---

## 2. Key Variables — What We Chose and Why

We started with 6 candidate variables, computed per circuit.

| Variable | What it measures | Why we considered it |
|---|---|---|
| `accident_rate` | Share of results that ended in a crash-related status (Accident, Collision, Spun off, Fatal accident, Collision damage, Debris, Damage) | Most direct signal of circuit-caused danger |
| `mechanical_rate` | Share of results that ended in a mechanical failure (Engine, Gearbox, Brakes, etc.) | Could reflect how harsh a circuit is on cars (bumpy surface, heat, etc.) |
| `avg_laps_behind` | Average number of laps a *finisher* was behind the leader (`+N Laps` statuses) | Captures cases where a driver technically "finished" but was far off competitive pace — a signal `is_finished` alone misses |
| `avg_position_change` | Average absolute difference between grid position and finish position | Directly measures how much race results deviate from qualifying expectations |
| `laptime_std` | Standard deviation of lap times per circuit | Higher variability could indicate more incidents, safety car periods, or pace disruption |
| `avg_pitstops` | Average pit stops per race per circuit | Could reflect tire wear or unexpected race incidents (debris, punctures) |

### How each was calculated

```python
# Status categorization (grouping raw status strings into buckets)
def categorize_status(status):
    if status == 'Finished':
        return 'finished_clean'
    elif status.startswith('+') and 'Lap' in status:
        return 'finished_behind'
    elif status in ['Accident', 'Collision', 'Spun off', 'Fatal accident',
                    'Collision damage', 'Debris', 'Damage']:
        return 'accident'
    elif status in ['Disqualified', 'Withdrew', 'Not classified', '107% Rule',
                    'Did not qualify', 'Did not prequalify', 'Excluded',
                    'Injury', 'Injured', 'Illness', 'Driver unwell',
                    'Safety concerns', 'Not restarted', 'Underweight',
                    'Safety belt', 'Safety', 'Eye injury']:
        return 'administrative'
    else:
        return 'mechanical'

# accident_rate / mechanical_rate = category count / total results, per circuit
# avg_laps_behind = mean of laps extracted from '+N Lap(s)' status strings
# avg_position_change = mean(abs(grid - position))
# laptime_std = std of lap_times.milliseconds, joined to circuit via raceId
# avg_pitstops = mean count of pit_stops rows per race, joined to circuit via raceId
```

### Why we dropped `laptime_std` and `avg_pitstops`

A first PCA run (all 6 features, all-time data) showed these two contributed
almost nothing:

| Feature | Loading (PC1) |
|---|---|
| accident_rate | 0.140 |
| mechanical_rate | 0.601 |
| avg_laps_behind | 0.503 |
| avg_position_change | 0.580 |
| laptime_std | 0.174 |
| avg_pitstops | -0.020 |

Explained variance: **41.6%**

We re-ran PCA with only the 4 strongest features (`accident_rate`,
`mechanical_rate`, `avg_laps_behind`, `avg_position_change`):

- Explained variance improved to **54.2%**
- Correlation between the 6-feature score and the 4-feature score: **0.99**

This confirmed `laptime_std` and `avg_pitstops` were adding noise rather than
signal, and were dropped from the final model. (Separately, both variables
also had heavy missing data in older eras — `lap_times.csv` and
`pit_stops.csv` only contain modern-era records — which made them
incompatible with a phase-based analysis anyway.)

---

## 3. Why PCA — and How It Was Used

### The problem
We had 4 variables per circuit and wanted to combine them into a single
"score." A manual weighted sum (like we used for the driver consistency
score) requires deciding weights by hand, which is subjective.

### The solution: PCA (Principal Component Analysis)
PCA finds the direction in the data along which circuits differ from each
other the most, and expresses each circuit as a single value along that
direction (the first principal component, PC1). The "weight" each original
variable gets in that combined score (the **loading**) is derived
mathematically from the data's variance structure, not chosen by hand.

### Steps taken
```python
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = ['accident_rate', 'mechanical_rate', 'avg_laps_behind', 'avg_position_change']

# 1. Standardize (mean 0, std 1) so no feature dominates due to scale
scaled = StandardScaler().fit_transform(circuit_stats[feature_cols])

# 2. Reduce to a single score
pca = PCA(n_components=1)
circuit_stats['difficulty_score'] = pca.fit_transform(scaled)[:, 0]

# 3. Inspect which variables drove the score
loadings = pd.DataFrame(pca.components_.T, columns=['PC1'], index=feature_cols)
```

### All-time (no phase split) result

Explained variance: 54.2%. Top 10 circuits by score:

| Circuit | Score |
|---|---|
| Zeltweg | 4.74 |
| Circuit de Pedralbes | 2.34 |
| Reims-Gueux | 2.24 |
| Indianapolis Motor Speedway | 2.13 |
| Phoenix street circuit | 1.98 |
| Donington Park | 1.87 |
| Detroit Street Circuit | 1.80 |
| Aintree | 1.76 |
| Montjuïc | 1.59 |
| Brands Hatch | 1.46 |

**Problem identified**: this list is dominated almost entirely by circuits
from F1's earliest era (1950s–60s), and notably excludes circuits widely
regarded as difficult today (e.g. Monaco). This pointed to a confound: the
score was capturing *how dangerous/unpredictable an era was*, not *how
difficult a specific circuit is*, since early-era cars were universally less
reliable and circuits had almost no safety infrastructure.

---

## 4. Why We Needed to Split by Era ("Safety Phase")

Comparing a 1960s race directly to a 2020s race is not a fair comparison —
car reliability and circuit safety standards were fundamentally different.
Without splitting by era, "circuit difficulty" and "how dangerous racing was
in general at that time" get mixed together and can't be separated.

### Phase definition

```python
def get_safety_phase(year):
    # before 1978: almost no circuit safety standards
    # 1978-1993: gradual safety improvements (post Ronnie Peterson fatal crash),
    #            but still the high-risk ground-effect/turbo era
    # 1994+: FIA overhaul of circuit and car safety after the Senna/Ratzenberger
    #        fatal accidents at Imola
    if year < 1978:
        return 'badSafety'
    elif year < 1994:
        return 'normalSafety'
    else:
        return 'goodSafety'
```

### Minimum sample size filter
To avoid a circuit's score being distorted by just 1–2 races, we required at
least 5 races within a given phase for a (phase, circuit) pair to be
included:

```python
MIN_RACES = 5
valid_circuits = phase_stats[phase_stats['race_count'] >= MIN_RACES]
```

### Re-running PCA within each phase
The same 4-feature PCA was re-run **separately for each phase**, so circuits
are only compared against others from the same era:

```python
for phase in circuit_stats_phase['safety_phase'].unique():
    phase_data = circuit_stats_phase[circuit_stats_phase['safety_phase'] == phase].copy()
    scaled = StandardScaler().fit_transform(phase_data[feature_cols])
    pca_phase = PCA(n_components=1)
    phase_data['difficulty_score'] = pca_phase.fit_transform(scaled)[:, 0]
```

---

## 5. Results After Splitting by Phase

### badSafety (pre-1978) — Top 10
| Circuit | Score |
|---|---|
| Indianapolis Motor Speedway | 2.27 |
| Mosport International Raceway | 1.74 |
| Autódromo José Carlos Pace | 1.56 |
| Circuit de Monaco | 1.56 |
| Red Bull Ring | 1.31 |
| Jarama | 1.18 |
| Nürburgring | 0.77 |
| Scandinavian Raceway | 0.41 |
| Circuit Park Zandvoort | 0.34 |
| Kyalami | 0.10 |

### goodSafety (1994+) — Top 10
| Circuit | Score |
|---|---|
| Indianapolis Motor Speedway | 3.28 |
| Circuit Gilles Villeneuve | 2.00 |
| Hockenheimring | 1.79 |
| Albert Park Grand Prix Circuit | 1.76 |
| Circuit de Monaco | 1.75 |
| Autodromo Enzo e Dino Ferrari | 1.55 |
| Autódromo José Carlos Pace | 1.34 |
| Nürburgring | 1.30 |
| Sepang International Circuit | 1.22 |
| Circuit de Nevers Magny-Cours | 0.92 |

### normalSafety (1978–1993) — Top 10
| Circuit | Score |
|---|---|
| Long Beach | 3.76 |
| Zolder | 1.51 |
| Circuit de Monaco | 1.01 |
| Detroit Street Circuit | 0.97 |
| Hungaroring | 0.86 |
| Autódromo Internacional Nelson Piquet | 0.80 |
| Suzuka Circuit | 0.77 |
| Adelaide Street Circuit | 0.60 |
| Kyalami | 0.54 |
| Autódromo Hermanos Rodríguez | 0.38 |

### What improved
- Circuits are now compared fairly within their own era, removing the
  "1950s circuits dominate everything" distortion.
- Monaco consistently appears across all three phases (rank 3–5), which
  aligns with its real-world reputation as a persistently tricky circuit.
- Indianapolis Motor Speedway ranks #1 in two separate eras — a genuinely
  interesting, era-independent finding (likely due to being an oval-track
  venue retrofitted for F1, unlike purpose-built road circuits).

### Remaining open questions
- Long Beach's outlier score (3.76, far above 2nd place) needs verification
  against its actual race count in that phase — could be a small-sample
  distortion rather than a real signal.
- Some historically notorious circuits (e.g. the old Nürburgring
  Nordschleife, old Spa-Francorchamps) don't appear prominently — this may
  be because they didn't meet the `MIN_RACES = 5` threshold in that phase,
  not because the metric judged them "safe."
- `mechanical_rate` still appears to be a major driver of the score in most
  phases, which raises the question of whether we're measuring circuit
  danger or car reliability at that venue.

In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# 1. Load raw data files
# ============================================================
df_results = pd.read_csv('../data/results.csv')
df_races = pd.read_csv('../data/races.csv')
df_circuits = pd.read_csv('../data/circuits.csv')
df_status = pd.read_csv('../data/status.csv')
df_lap_times = pd.read_csv('../data/lap_times.csv')
df_pit_stops = pd.read_csv('../data/pit_stops.csv')

# ============================================================
# 2. Merge results + races + circuits + status into one table
# ============================================================
results_subset = df_results[['raceId', 'grid', 'position', 'statusId']]
races_subset = df_races[['raceId', 'circuitId', 'year']]
circuits_subset = df_circuits[['circuitId', 'name']].rename(columns={'name': 'circuitName'})

df = results_subset.merge(races_subset, on='raceId', how='left')
df = df.merge(circuits_subset, on='circuitId', how='left')
df = df.merge(df_status, on='statusId', how='left')


# ============================================================
# 3. Categorize status into: finished_clean, finished_behind,
#    accident, administrative, mechanical
# ============================================================
def categorize_status(status):
    if status == 'Finished':
        return 'finished_clean'
    elif status.startswith('+') and 'Lap' in status:
        return 'finished_behind'
    elif status in ['Accident', 'Collision', 'Spun off', 'Fatal accident',
                    'Collision damage', 'Debris', 'Damage']:
        return 'accident'
    elif status in ['Disqualified', 'Withdrew', 'Not classified', '107% Rule',
                    'Did not qualify', 'Did not prequalify', 'Excluded',
                    'Injury', 'Injured', 'Illness', 'Driver unwell',
                    'Safety concerns', 'Not restarted', 'Underweight',
                    'Safety belt', 'Safety', 'Eye injury']:
        return 'administrative'
    else:
        return 'mechanical'


df['status_category'] = df['status'].apply(categorize_status)


# ============================================================
# 4. Derived columns: laps_behind, position_change
# ============================================================
def extract_laps_behind(status):
    match = re.match(r'^\+(\d+) Lap', status)
    if match:
        return int(match.group(1))
    elif status == 'Finished':
        return 0
    else:
        return np.nan


df['laps_behind'] = df['status'].apply(extract_laps_behind)
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['position_change'] = abs(df['grid'] - df['position'])

# ============================================================
# 5. Aggregate per circuit (ALL-TIME, no phase split)
# ============================================================
category_counts = df.groupby(['circuitName', 'status_category']).size().unstack(fill_value=0)
category_counts['total'] = category_counts.sum(axis=1)
category_counts['accident_rate'] = category_counts['accident'] / category_counts['total']
category_counts['mechanical_rate'] = category_counts['mechanical'] / category_counts['total']

avg_laps_behind = df.groupby('circuitName')['laps_behind'].mean()
avg_position_change = df.groupby('circuitName')['position_change'].mean()

lap_join = df_lap_times.merge(df[['raceId', 'circuitName']].drop_duplicates(), on='raceId', how='left')
laptime_std = lap_join.groupby('circuitName')['milliseconds'].std()

pit_join = df_pit_stops.merge(df[['raceId', 'circuitName']].drop_duplicates(), on='raceId', how='left')
pitstop_count = pit_join.groupby(['circuitName', 'raceId']).size().groupby('circuitName').mean()

circuit_stats = category_counts[['accident_rate', 'mechanical_rate', 'total']].reset_index()
circuit_stats = circuit_stats.merge(avg_laps_behind.reset_index(name='avg_laps_behind'), on='circuitName')
circuit_stats = circuit_stats.merge(avg_position_change.reset_index(name='avg_position_change'), on='circuitName')
circuit_stats = circuit_stats.merge(laptime_std.reset_index(name='laptime_std'), on='circuitName', how='left')
circuit_stats = circuit_stats.merge(pitstop_count.reset_index(name='avg_pitstops'), on='circuitName', how='left')

# Drop circuits with too few results to trust the statistics
circuit_stats = circuit_stats[circuit_stats['total'] >= 20]

# ============================================================
# 6. v1: PCA with all 6 features
# ============================================================
feature_cols_v1 = ['accident_rate', 'mechanical_rate', 'avg_laps_behind',
                    'avg_position_change', 'laptime_std', 'avg_pitstops']

circuit_stats_clean = circuit_stats.dropna(subset=feature_cols_v1).copy()

scaled = StandardScaler().fit_transform(circuit_stats_clean[feature_cols_v1])

pca = PCA(n_components=1)
circuit_stats_clean['difficulty_score'] = pca.fit_transform(scaled)[:, 0]

loadings = pd.DataFrame(pca.components_.T, columns=['PC1'], index=feature_cols_v1)
print("=== v1: all 6 features ===")
print(loadings)
print("Explained variance:", pca.explained_variance_ratio_)
print()

# ============================================================
# 7. v2: PCA with 4 features (dropped avg_pitstops, laptime_std -
#    both had near-zero loadings in v1)
# ============================================================
feature_cols_v2 = ['accident_rate', 'mechanical_rate', 'avg_laps_behind', 'avg_position_change']

circuit_stats_clean_v2 = circuit_stats.dropna(subset=feature_cols_v2).copy()

scaled_v2 = StandardScaler().fit_transform(circuit_stats_clean_v2[feature_cols_v2])

pca_v2 = PCA(n_components=1)
circuit_stats_clean_v2['difficulty_score_v2'] = pca_v2.fit_transform(scaled_v2)[:, 0]

loadings_v2 = pd.DataFrame(pca_v2.components_.T, columns=['PC1'], index=feature_cols_v2)
print("=== v2: 4 features (dropped avg_pitstops, laptime_std) ===")
print(loadings_v2)
print("Explained variance:", pca_v2.explained_variance_ratio_)
print()

# ============================================================
# 8. Compare v1 vs v2 rankings (are they still similar?)
# ============================================================
comparison = circuit_stats_clean_v2[['circuitName', 'difficulty_score_v2']].merge(
    circuit_stats_clean[['circuitName', 'difficulty_score']], on='circuitName'
)
print("Correlation between v1 and v2 scores:")
print(comparison[['difficulty_score_v2', 'difficulty_score']].corr())
print()

# ============================================================
# 9. Top 10 circuits (all-time, v2 - the version we ended up trusting)
# ============================================================
print("=== All-time Top 10 (v2, no phase split) ===")
print(
    circuit_stats_clean_v2.sort_values('difficulty_score_v2', ascending=False)
    [['circuitName', 'difficulty_score_v2']].head(10)
)

In [ ]:
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# 1. Load raw data files
# ============================================================
df_results = pd.read_csv('../data/results.csv')
df_races = pd.read_csv('../data/races.csv')
df_circuits = pd.read_csv('../data/circuits.csv')
df_status = pd.read_csv('../data/status.csv')
df_lap_times = pd.read_csv('../data/lap_times.csv')
df_pit_stops = pd.read_csv('../data/pit_stops.csv')

# ============================================================
# 2. Merge results + races + circuits + status into one table
# ============================================================
results_subset = df_results[['raceId', 'grid', 'position', 'statusId']]
races_subset = df_races[['raceId', 'circuitId', 'year']]
circuits_subset = df_circuits[['circuitId', 'name']].rename(columns={'name': 'circuitName'})

df = results_subset.merge(races_subset, on='raceId', how='left')
df = df.merge(circuits_subset, on='circuitId', how='left')
df = df.merge(df_status, on='statusId', how='left')


# ============================================================
# 3. Categorize status into: finished_clean, finished_behind,
#    accident, administrative, mechanical
# ============================================================
def categorize_status(status):
    if status == 'Finished':
        return 'finished_clean'
    elif status.startswith('+') and 'Lap' in status:
        return 'finished_behind'
    elif status in ['Accident', 'Collision', 'Spun off', 'Fatal accident',
                    'Collision damage', 'Debris', 'Damage']:
        return 'accident'
    elif status in ['Disqualified', 'Withdrew', 'Not classified', '107% Rule',
                    'Did not qualify', 'Did not prequalify', 'Excluded',
                    'Injury', 'Injured', 'Illness', 'Driver unwell',
                    'Safety concerns', 'Not restarted', 'Underweight',
                    'Safety belt', 'Safety', 'Eye injury']:
        return 'administrative'
    else:
        return 'mechanical'


df['status_category'] = df['status'].apply(categorize_status)


# ============================================================
# 4. Derived columns: laps_behind, position_change
# ============================================================
def extract_laps_behind(status):
    match = re.match(r'^\+(\d+) Lap', status)
    if match:
        return int(match.group(1))
    elif status == 'Finished':
        return 0
    else:
        return np.nan


df['laps_behind'] = df['status'].apply(extract_laps_behind)
df['position'] = pd.to_numeric(df['position'], errors='coerce')
df['position_change'] = abs(df['grid'] - df['position'])


# ============================================================
# 5. Assign safety phase based on year
# ============================================================
def get_safety_phase(year):
    # before 1978: almost no circuit safety standards
    # 1978-1993: gradual safety improvements (post Ronnie Peterson crash), still risky turbo/ground-effect era
    # 1994+: FIA overhaul after Senna/Ratzenberger fatal accidents at Imola
    if year < 1978:
        return 'badSafety'
    elif year < 1994:
        return 'normalSafety'
    else:
        return 'goodSafety'


df['safety_phase'] = df['year'].apply(get_safety_phase)

# ============================================================
# 6. Filter to circuits with enough races within each phase
# ============================================================
phase_stats = df.groupby(['safety_phase', 'circuitName'])['raceId'].nunique().reset_index(name='race_count')

MIN_RACES = 5
valid_circuits = phase_stats[phase_stats['race_count'] >= MIN_RACES]

# ============================================================
# 7. Compute accident_rate, mechanical_rate, avg_laps_behind, avg_position_change
#    grouped by (safety_phase, circuitName)
# ============================================================
category_counts_phase = df.groupby(['safety_phase', 'circuitName', 'status_category']).size().unstack(fill_value=0)
category_counts_phase['total'] = category_counts_phase.sum(axis=1)
category_counts_phase['accident_rate'] = category_counts_phase['accident'] / category_counts_phase['total']
category_counts_phase['mechanical_rate'] = category_counts_phase['mechanical'] / category_counts_phase['total']

avg_laps_behind_phase = df.groupby(['safety_phase', 'circuitName'])['laps_behind'].mean()
avg_position_change_phase = df.groupby(['safety_phase', 'circuitName'])['position_change'].mean()

circuit_stats_phase = category_counts_phase[['accident_rate', 'mechanical_rate']].reset_index()
circuit_stats_phase = circuit_stats_phase.merge(
    avg_laps_behind_phase.reset_index(name='avg_laps_behind'), on=['safety_phase', 'circuitName']
)
circuit_stats_phase = circuit_stats_phase.merge(
    avg_position_change_phase.reset_index(name='avg_position_change'), on=['safety_phase', 'circuitName']
)

# ============================================================
# 8. Compute laptime_std and avg_pitstops grouped by (safety_phase, circuitName)
# ============================================================
race_phase_lookup = df[['raceId', 'circuitName', 'safety_phase']].drop_duplicates()

lap_join_phase = df_lap_times.merge(race_phase_lookup, on='raceId', how='left')
laptime_std_phase = lap_join_phase.groupby(['safety_phase', 'circuitName'])['milliseconds'].std()

pit_join_phase = df_pit_stops.merge(race_phase_lookup, on='raceId', how='left')
pitstop_count_phase = (
    pit_join_phase.groupby(['safety_phase', 'circuitName', 'raceId']).size()
    .groupby(['safety_phase', 'circuitName']).mean()
)

circuit_stats_phase_full = circuit_stats_phase.merge(
    laptime_std_phase.reset_index(name='laptime_std'), on=['safety_phase', 'circuitName'], how='left'
)
circuit_stats_phase_full = circuit_stats_phase_full.merge(
    pitstop_count_phase.reset_index(name='avg_pitstops'), on=['safety_phase', 'circuitName'], how='left'
)

# ============================================================
# 9. Keep only (phase, circuit) combos with enough races
# ============================================================
circuit_stats_phase_full = circuit_stats_phase_full.merge(
    valid_circuits[['safety_phase', 'circuitName']],
    on=['safety_phase', 'circuitName'],
    how='inner'
)

print("Missing values per column:")
print(circuit_stats_phase_full.isnull().sum())
print()

# ============================================================
# 10. Run PCA separately within each safety phase, using all 6 features
# ============================================================
feature_cols_all6 = ['accident_rate', 'mechanical_rate', 'avg_laps_behind',
                    'avg_position_change']

results_by_phase = []

for phase in circuit_stats_phase_full['safety_phase'].unique():
    phase_data = circuit_stats_phase_full[circuit_stats_phase_full['safety_phase'] == phase].copy()
    phase_data_clean = phase_data.dropna(subset=feature_cols_all6)

    if len(phase_data_clean) < 5:
        print(f"{phase}: skipped, not enough complete rows ({len(phase_data_clean)})")
        continue

    scaled = StandardScaler().fit_transform(phase_data_clean[feature_cols_all6])

    pca_phase = PCA(n_components=1)
    phase_data_clean['difficulty_score'] = pca_phase.fit_transform(scaled)[:, 0]

    loadings = pd.DataFrame(pca_phase.components_.T, columns=['PC1'], index=feature_cols_all6)
    print(f"=== {phase} (n={len(phase_data_clean)}) ===")
    print(loadings)
    print("Explained variance:", pca_phase.explained_variance_ratio_)
    print()

    results_by_phase.append(phase_data_clean)

circuit_stats_final = pd.concat(results_by_phase, ignore_index=True)

# ============================================================
# 11. Show top 5 most "difficult" circuits per phase
# ============================================================
for phase in circuit_stats_final['safety_phase'].unique():
    print(f"=== {phase} Top 10 ===")
    top5 = circuit_stats_final[circuit_stats_final['safety_phase'] == phase] \
        .sort_values('difficulty_score', ascending=False).head(10)
    print(top5[['circuitName', 'difficulty_score']])
    print()

# Task14 - [Advanced] Driver Performance Trend (Multi-Year)

## Driver Career Arc Interpretations

### Lewis Hamilton (2007–2024)
Hamilton's normalized trend shows a clear step-change in 2013–2014, coinciding with his move from McLaren to Mercedes just as the sport transitioned to the V6 hybrid turbo era. From 2014 to 2020, he sustained a remarkably high and stable points percentage (roughly 0.75–0.85), reflecting sustained dominance rather than a single peak season. The sharp decline from 2021 onward—dropping below 0.45—aligns with Red Bull's rise and suggests his career arc is less a gradual decline and more an abrupt loss of competitive machinery, since his output had been consistently elite for nearly a decade beforehand.

### Max Verstappen (2015–2024)
Verstappen's arc is a textbook "rise to dominance." After a modest debut with Toro Rosso (2015) and early Red Bull years hovering around 0.35–0.55, his trajectory turns sharply upward from 2021, peaking near 0.95 in 2023 — one of the highest values in the dataset. Unlike Hamilton's plateau, Verstappen's rise appears tightly linked to Red Bull's competitive resurgence rather than personal development alone, given how closely the points percentage tracks his team's form. The slight dip in 2024 raises an open question of whether this marks the beginning of a decline or a temporary dip.

### Fernando Alonso (2001–2024)
Alonso's career shows the most distinct multi-peak pattern of the three. His first peak (2005–2006, Renault) and second peak (2010–2013, Ferrari) are separated by a clear trough, and both are followed by long stretches of underperformance (2015–2020 at McLaren, largely below 0.15) that align with uncompetitive machinery rather than a decline in ability. The 2023 spike at Aston Martin suggests he remains capable of near-peak performance well into his 40s when given a strong car, reinforcing that his fluctuations track team competitiveness more than personal aging.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Step 0: Load data and confirm actual column names
# ============================================================
final = pd.read_csv('../data/merged_f1.csv')
DRIVER_COL = 'full_name'
TEAM_COL = 'name'  

# ============================================================
# Step 1: Pick 3 drivers and get their season-by-season points
# ============================================================
drivers_to_plot = ['Lewis Hamilton', 'Max Verstappen', 'Fernando Alonso']

driver_trends = final[final[DRIVER_COL].isin(drivers_to_plot)]
season_points = driver_trends.groupby([DRIVER_COL, 'year'])['points'].sum().reset_index()

# Confirm each driver has at least 8 seasons of data
print(season_points.groupby(DRIVER_COL)['year'].nunique())

In [ ]:
# ============================================================
# Step 2: Points-system normalization (account for rule changes over eras)
# ============================================================
# f1_points_systems and get_points_system() were defined earlier in the project
races_per_year = final.groupby('year')['raceId'].nunique()

def get_win_points(year):
    system = get_points_system(year)
    return system[1]   # points awarded for 1st place that year

season_points['win_points'] = season_points['year'].apply(get_win_points)
season_points['race_count'] = season_points['year'].map(races_per_year)
season_points['max_possible'] = season_points['win_points'] * season_points['race_count']
season_points['points_pct'] = season_points['points'] / season_points['max_possible']

print(season_points[[DRIVER_COL, 'year', 'points', 'max_possible', 'points_pct']].head(10))

In [ ]:
# ============================================================
# Step 3: Determine each driver's main team per season (handles mid-season switches)
# ============================================================
def get_main_team(group):
    return group[TEAM_COL].mode()[0]   # most frequent team that season

season_main_team = (
    driver_trends.groupby([DRIVER_COL, 'year'])
    .apply(get_main_team)
    .reset_index(name='main_team')
)

season_points = season_points.merge(season_main_team, on=[DRIVER_COL, 'year'], how='left')

In [ ]:
# ============================================================
# Step 4: Detect team-change years automatically
# ============================================================
season_points = season_points.sort_values([DRIVER_COL, 'year'])
season_points['prev_team'] = season_points.groupby(DRIVER_COL)['main_team'].shift(1)
season_points['team_changed'] = season_points['main_team'] != season_points['prev_team']

In [ ]:
# ============================================================
# Step 5: Plot raw points and normalized points side by side,
#         with team-change markers
# ============================================================
def add_team_markers(ax, driver_data, driver, color):
    # Mark the starting team (first season) separately
    first_row = driver_data.iloc[0]
    ax.axvline(x=first_row['year'], color=color, linestyle=':', alpha=0.4)
    ax.text(
        first_row['year'], ax.get_ylim()[1] * 0.5, f"{first_row['main_team']} (debut)",
        rotation=90, fontsize=8, color='white', va='center', ha='center',
        bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=1.5)
    )

    # Mark every subsequent team change
    changes = driver_data[(driver_data['team_changed']) & (driver_data['prev_team'].notna())]
    for i, (_, row) in enumerate(changes.iterrows()):
        ax.axvline(x=row['year'], color=color, linestyle='--', alpha=0.4)
        # Alternate vertical position slightly so labels don't overlap
        y_frac = 0.85 if i % 2 == 0 else 0.6
        ax.text(
            row['year'], ax.get_ylim()[1] * y_frac, row['main_team'],
            rotation=90, fontsize=8, color='white', va='center', ha='center',
            bbox=dict(facecolor=color, alpha=0.7, edgecolor='none', pad=1.5)
        )


fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for driver in drivers_to_plot:
    driver_data = season_points[season_points[DRIVER_COL] == driver]

    axes[0].plot(driver_data['year'], driver_data['points'], marker='o', label=driver, color=colors[driver])
    axes[1].plot(driver_data['year'], driver_data['points_pct'], marker='o', label=driver, color=colors[driver])

    add_team_markers(axes[0], driver_data, driver, colors[driver])
    add_team_markers(axes[1], driver_data, driver, colors[driver])

axes[0].set_title("Raw Points per Season")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Points")
axes[0].legend(loc='upper left')

axes[1].set_title("Normalized Points (% of max possible)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Points %")
axes[1].legend(loc='upper left')

plt.tight_layout()
plt.savefig('chart4_driver_trends.png', dpi=300, bbox_inches='tight')
plt.show()

# Task15 - [Advanced] Predictive Mini-Model (Optional Stretch)
# F1 Top10 Finish Prediction Model — Logistic Regression Project

This document summarizes the process of building a classification model to predict
whether a driver finishes in the Top 10 in a given race. It covers the math, the
rationale behind feature selection, why `fastf1` was used, and the full pipeline.

---

## 1. Problem Definition

> **Using features such as grid position, constructor, and circuit, build a model
> that predicts whether a driver finishes in the Top 10 for a given race, then
> evaluate accuracy on a held-out test set.**

- **Target (Y)**: Binary classification. `1` = finished Top 10, `0` = finished outside Top 10 or DNF
- **Model**: Logistic regression (no deep learning needed, chosen for interpretability)
- **Training data**: 2020–2024 seasons
- **Test data**: 2025 season (data the model has never seen)

---

## 2. Logistic Regression — The Math

### Why logistic regression instead of plain linear regression

Since Y is not a continuous number (like points or position) but strictly **0 or 1**,
using linear regression directly could produce predictions outside the 0–1 range.
Logistic regression uses a sigmoid function to always compress predictions into a
**probability between 0 and 1**.

### Sigmoid Function

The linear combination $z$ is transformed into a probability:

$$
z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n
$$

$$
P(\text{Top10} = 1 \mid X) = \sigma(z) = \frac{1}{1 + e^{-z}}
$$

- $\beta_0$: intercept
- $\beta_1, \dots, \beta_n$: coefficients for each feature — learned from the data
- $\sigma(z)$: always between 0 and 1, interpretable as a probability

### Final Prediction (using a 0.5 threshold)

$$
\hat{y} =
\begin{cases}
1 & \text{if } P(\text{Top10}=1 \mid X) \geq 0.5 \\
0 & \text{otherwise}
\end{cases}
$$

### How the Model Learns — Maximizing Log-Likelihood

Logistic regression finds the $\beta$ values that maximize the log-likelihood:

$$
\ell(\beta) = \sum_{i=1}^{n} \Big[ y_i \log(p_i) + (1 - y_i)\log(1 - p_i) \Big]
$$

- $y_i$: the actual label (0 or 1)
- $p_i$: the probability predicted by the model

### McFadden's Pseudo R²

Logistic regression doesn't have a direct equivalent of the standard regression
$R^2$, so this pseudo-metric is used instead:

$$
\text{Pseudo } R^2 = 1 - \frac{\ell_{\text{model}}}{\ell_{\text{null}}}
$$

- $\ell_{\text{model}}$: log-likelihood of our fitted model
- $\ell_{\text{null}}$: log-likelihood of a "null" model that predicts only the overall base rate
- **Interpretation note**: don't expect values like 0.7–0.9 as with regression $R^2$ —
  **0.2–0.4 is already considered a reasonably good fit** for this metric

---

## 3. Why We Used `fastf1`

### The problem
The existing Ergast-based CSV dataset only went up through the 2024 season, meaning
**no 2025 test data existed** within it. The dataset also had **no weather information
at all**, so a separate data source was needed to include weather as a feature.

### What fastf1 solved
`fastf1` is an open-source Python library that pulls official F1 live timing data. It provides:
- **Race results (grid, finishing position, status) from the 2018 season onward**
- **Per-session weather data (air temperature, track temperature, humidity, rainfall)**
  bundled together, with no separate weather API call needed

This let us solve both the "missing 2025 test set" problem and the "missing weather
data" problem **with a single library**.

```python
import fastf1

fastf1.Cache.enable_cache('./f1_cache')  # avoid re-downloading on repeated runs

session = fastf1.get_session(2025, 'Australia', 'R')
session.load(weather=True)

session.results       # grid, position, status, points
session.weather_data  # AirTemp, TrackTemp, Humidity, Rainfall
```

### Practical considerations
- **API rate limit (500 calls/hour)**: results were saved race-by-race to disk, with
  `os.path.exists()` checks to skip already-downloaded races — so the process could
  safely resume after hitting the limit
- **`time.sleep()` between requests** to avoid triggering the rate limit in the first place

---

## 4. Feature Selection — What Was Chosen and Why

### Three selection criteria
1. **No data leakage**: never use information that wouldn't actually be known before
   the race starts (e.g. that race's own finishing status or points)
2. **Avoid multicollinearity**: when two features represent the same underlying concept,
   keep only one
3. **Prefer raw facts over derived claims**: favor original data or simple aggregations
   over our own composite metrics (e.g. the PCA-based circuit difficulty score)

### Final selected features and rationale

| Feature | Definition | Why it was chosen |
|---|---|---|
| `grid` | Qualifying result (starting position) | Already validated correlation with finishing position (r=0.638). The most direct predictive signal |
| `constructor_season_points_so_far` | Team's cumulative points that season, **before this race** | Represents "current team competitiveness" as a continuous number instead of a categorical team name; also captures in-season performance shifts |
| `driver_circuit_avg_position` | This driver's average finishing position at this circuit, **prior to this race** | A more direct signal than "is this circuit hard in general" — captures whether this specific driver performs well at this specific venue. The earlier PCA-based circuit difficulty score was dropped due to validated limitations |
| `driver_recent_form` | Average finishing position over the last 3 races (excluding this one) | Captures recent momentum/condition |
| `driver_dnf_rate_recent` | DNF rate over the last 5 races (excluding this one) | Reliability, a distinct dimension from finishing position — no multicollinearity concern |
| `teammate_grid_position` | Teammate's grid position in this same race | A short-term signal of "is this team's car well set up this weekend" (different time scale from season-long team points) |
| `driver_experience` | Cumulative career races entered | Chosen over driver age — more directly tied to the "adaptation period" reasoning |
| `air_temp`, `track_temp`, `rainfall` | Average temperature/track temperature/rain during the race | Tests the hypothesis that weather affects how predictable a race outcome is |

### Features excluded and why

| Excluded feature | Reason |
|---|---|
| `points`, `position`, `positionOrder` (this race) | Nearly identical to the prediction target (Y) → data leakage |
| `statusId`, `laps` (this race) | Only knowable after the race ends |
| `constructorName` (raw team name) | Conceptually redundant with `constructor_season_points_so_far` (multicollinearity) |
| `circuitName` (raw circuit name) | Replaced by `driver_circuit_avg_position` — the driver-circuit interaction was judged more meaningful |
| `driver_career_avg_position`, `driver_age` | Conceptually redundant with `driver_recent_form` and `driver_experience` respectively |
| `circuit_overtaking_difficulty` (PCA-based) | Its own validation revealed limitations (small sample sizes, era confounds) → excluded as a "constructed claim" in favor of fact-based features |
| `safety_phase` | Since only 2020–2025 is used, every row falls into the same phase — zero variance, no discriminative value |

---

## 5. Preventing Data Leakage — Using `shift()`

When engineering derived features, always shift by one so that the current race's
own result never leaks into its own predictors:

```python
# Correct: excludes the current race, uses only prior races
df['driver_recent_form'] = (
    df.groupby('full_name')['position']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)
```

Without `shift(1)`, the model would end up using this race's own result to predict
this race's own result — a circular logic error.

---

## 6. Forward Selection — Validating Features One at a Time

```python
def forward_selection(X, y, candidate_cols):
    selected = []
    remaining = list(candidate_cols)
    best_score_overall = 0

    while remaining:
        scores = []
        for col in remaining:
            model = LogisticRegression(max_iter=1000)
            model.fit(X[selected + [col]], y)
            scores.append((model.score(X[selected + [col]], y), col))

        scores.sort(reverse=True)
        best_score, best_col = scores[0]

        if best_score > best_score_overall:
            best_score_overall = best_score
            selected.append(best_col)
            remaining.remove(best_col)
        else:
            break

    return selected
```

**Logic**: start with zero features → add whichever candidate improves performance
the most → stop once no further improvement is found. This validates each feature's
usefulness empirically rather than by intuition alone.

---

## 7. Full Pipeline Summary

```
1. Fetch 2020-2025 season results + weather via fastf1 (per-race saving, caching)
2. Standardize column names (FullName -> full_name, etc.)
3. Create Y: position <= 10 -> 1, else 0 (DNFs automatically become 0)
4. Engineer features (all using shift(1) to prevent data leakage)
   - driver_recent_form, driver_dnf_rate_recent, driver_experience
   - driver_circuit_avg_position, constructor_season_points_so_far
   - teammate_grid_position
5. Split into Train (2020-2024) / Test (2025)
6. Run forward selection to decide which features to keep
7. Train the final logistic regression model on selected features
8. Evaluate on the 2025 test set: Accuracy, Confusion Matrix, McFadden's Pseudo R^2
9. Write a prediction function for 2026 (driver name + circuit + grid + weather -> Top10 probability)
```

---

## 8. Limitations

### Test set results (2025 held-out season)

```
Accuracy: 0.7584
Confusion Matrix:
              Predicted 0    Predicted 1
Actual 0         126             33
Actual 1          53            144

McFadden's Pseudo R^2 (train): 0.2875
McFadden's Pseudo R^2 (test):  0.2921
```

1. **`teammate_grid_position` must be estimated for future predictions**: at 2026
   prediction time, the teammate's grid position isn't known yet, so it's replaced
   with an average fallback value. If this feature was important in forward selection,
   this substitution meaningfully reduces real-world prediction accuracy.
2. **Weather is a forecast, not an observation, at prediction time**: training used
   actual observed weather, but future predictions must rely on forecast-level
   estimates, introducing additional error.
3. **Rookie drivers and new circuits have no history**, so fallback (overall average)
   values are used — likely causing under- or over-estimation of their true performance.
4. **A moderate Pseudo R² (~0.29) reflects the inherent unpredictability of race
   outcomes, not necessarily a weak model.** By McFadden's (1974) own guidance, a
   Pseudo R² in the 0.2–0.4 range is already considered an excellent fit for
   discrete choice models — this metric is not on the same scale as an OLS R²,
   and expecting values like 0.7–0.9 would be a misapplication of the benchmark.
   The fact that train (0.288) and test (0.292) Pseudo R² are nearly identical is
   a positive sign: the model generalizes to unseen 2025 data without overfitting.
5. **Pre-race features cannot capture in-race randomness.** Accidents, safety cars,
   mechanical failures, and mid-race strategy calls are, by design, excluded from
   this model (to avoid data leakage), yet they materially affect whether a driver
   finishes in the Top 10. This is very likely the dominant reason the Pseudo R²
   sits in the moderate range rather than higher — the model is not "missing" a
   feature so much as it is fundamentally limited by what is knowable before a
   race begins.
6. **Recall (73.1%) is somewhat lower than precision (81.4%)**, meaning the model
   is more conservative: it misses more true Top10 finishes (53 false negatives)
   than it wrongly flags non-Top10 finishes as Top10 (33 false positives). This
   suggests the model may be somewhat biased toward predicting "not Top10" in
   borderline cases, which could be worth addressing (e.g. via threshold tuning)
   if recall is prioritized over precision in a given use case.

In [ ]:
import fastf1
import pandas as pd
import numpy as np
import os
import time
import glob

# ============================================================
# Step 1: Enable caching (avoids re-downloading already-fetched data)
# ============================================================
CACHE_DIR = './f1_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

# ============================================================
# Step 2: Function to fetch and save one season, race by race
#          (skips races already saved, so it's safe to re-run
#          after hitting a rate limit)
# ============================================================
SAVE_DIR = '../data/raw_by_race'
os.makedirs(SAVE_DIR, exist_ok=True)

def get_season_data_safe(year, sleep_sec=3):
    schedule = fastf1.get_event_schedule(year)
    race_names = schedule[schedule['EventFormat'] != 'testing']['EventName'].tolist()

    for race_name in race_names:
        save_path = f"{SAVE_DIR}/{year}_{race_name.replace(' ', '_')}.csv"

        if os.path.exists(save_path):
            print(f"[skip] {year} {race_name} already saved")
            continue

        try:
            session = fastf1.get_session(year, race_name, 'R')
            session.load(weather=True)

            res = session.results.copy()
            res['year'] = year
            res['race_name'] = race_name

            # Average weather conditions across the race
            weather_summary = session.weather_data[['AirTemp', 'TrackTemp', 'Humidity', 'Rainfall']].mean()
            for col, val in weather_summary.items():
                res[col] = val

            res.to_csv(save_path, index=False)
            print(f"[ok] {year} {race_name} saved")

            time.sleep(sleep_sec)

        except Exception as e:
            print(f"[fail] {year} {race_name}: {e}")
            time.sleep(sleep_sec * 2)

# ============================================================
# Step 3: Run one season at a time, starting from 2020
# ============================================================
get_season_data_safe(2020)
get_season_data_safe(2021)
get_season_data_safe(2022)
get_season_data_safe(2023)
get_season_data_safe(2024)
get_season_data_safe(2025)

In [62]:
import pandas as pd
import numpy as np
import glob
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# ============================================================
# 0. Load and combine the fastf1-fetched race files
# ============================================================
RAW_DIR = '../data/raw_by_race'
TRAIN_START_YEAR = 2020   # change to 2021 if 2020 wasn't fetched yet
TRAIN_END_YEAR = 2024
TEST_YEAR = 2025

all_files = glob.glob(f'{RAW_DIR}/*.csv')
df_all = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

# Standardize column names to match the rest of the project
df_all = df_all.rename(columns={
    'FullName': 'full_name',
    'TeamName': 'constructorName',
    'GridPosition': 'grid',
    'Position': 'position',
    'Status': 'status',
    'AirTemp': 'air_temp',
    'TrackTemp': 'track_temp',
    'Humidity': 'humidity',
    'Rainfall': 'rainfall',
})

df_all['position'] = pd.to_numeric(df_all['position'], errors='coerce')
df_all['grid'] = pd.to_numeric(df_all['grid'], errors='coerce')

# Need a way to order races chronologically within/across seasons.
# fastf1's event schedule has a 'RoundNumber'; if not already present,
# derive an ordering from year + the order races were saved.
df_all = df_all.sort_values(['year', 'race_name']).reset_index(drop=True)
df_all['round_order'] = df_all.groupby('year').cumcount() if False else None
# Safer: rebuild round order using date if available, else just use appearance order
df_all['race_order_key'] = df_all['year'] * 100 + df_all.groupby('year')['race_name'].transform(
    lambda x: pd.factorize(x)[0]
)

# ============================================================
# 1. Target variable: Top 10 finish (1) vs not (0)
# ============================================================
df_all['top10'] = (df_all['position'] <= 10).astype(int)
# Drivers who DNF'd have NaN position -> correctly counted as 0 (not Top 10)
df_all['top10'] = np.where(df_all['position'].isna(), 0, df_all['top10'])

# ============================================================
# 2. Feature engineering (all features use ONLY past information,
#    never the current race's own result -> avoids data leakage)
# ============================================================
df_all = df_all.sort_values(['full_name', 'race_order_key'])

# --- driver_recent_form: mean finishing position over last 3 races (excluding current) ---
df_all['driver_recent_form'] = (
    df_all.groupby('full_name')['position']
    .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).mean())
)

# --- driver_dnf_rate_recent: DNF rate over last 5 races (excluding current) ---
df_all['dnf_flag'] = df_all['position'].isna().astype(int)
df_all['driver_dnf_rate_recent'] = (
    df_all.groupby('full_name')['dnf_flag']
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean())
)

# --- driver_experience: number of past races entered (excluding current) ---
df_all['driver_experience'] = df_all.groupby('full_name').cumcount()

# --- driver_circuit_avg_position: driver's historical average position at this circuit ---
df_all = df_all.sort_values(['full_name', 'race_name', 'race_order_key'])
df_all['driver_circuit_avg_position'] = (
    df_all.groupby(['full_name', 'race_name'])['position']
    .transform(lambda x: x.shift(1).expanding().mean())
)
df_all = df_all.sort_values(['full_name', 'race_order_key'])  # restore chronological order

# --- constructor_season_points_so_far: team's cumulative points within that season, before this race ---
df_all['Points'] = pd.to_numeric(df_all['Points'], errors='coerce').fillna(0)
df_all = df_all.sort_values(['constructorName', 'year', 'race_order_key'])
df_all['constructor_season_points_so_far'] = (
    df_all.groupby(['constructorName', 'year'])['Points']
    .transform(lambda x: x.shift(1).cumsum())
)
df_all['constructor_season_points_so_far'] = df_all['constructor_season_points_so_far'].fillna(0)

# --- teammate_grid_position: the OTHER driver on the same team, same race ---
def get_teammate_grid(group):
    if len(group) == 2:
        return group['grid'].values[::-1]   # swap the two grid values
    return [np.nan] * len(group)

df_all = df_all.sort_values(['year', 'race_name', 'constructorName'])
df_all['teammate_grid_position'] = (
    df_all.groupby(['year', 'race_name', 'constructorName'])
    .apply(lambda g: pd.Series(get_teammate_grid(g), index=g.index))
    .reset_index(level=[0, 1, 2], drop=True)
)

df_all = df_all.sort_values(['full_name', 'race_order_key']).reset_index(drop=True)

# --- weather features: use as-is (already race-level averages from fastf1) ---
# air_temp, track_temp, humidity, rainfall

# ============================================================
# 3. Assemble candidate feature list
# ============================================================
candidate_features = [
    'grid',
    'constructor_season_points_so_far',
    'driver_circuit_avg_position',
    'driver_recent_form',
    'driver_dnf_rate_recent',
    'teammate_grid_position',
    'driver_experience',
    'air_temp',
    'track_temp',
    'rainfall',
]

# ============================================================
# 4. Split into train (2020-2024) and test (2025)
# ============================================================
train_df = df_all[(df_all['year'] >= TRAIN_START_YEAR) & (df_all['year'] <= TRAIN_END_YEAR)].copy()
test_df = df_all[df_all['year'] == TEST_YEAR].copy()

train_df = train_df.dropna(subset=candidate_features + ['top10'])
test_df = test_df.dropna(subset=candidate_features + ['top10'])

X_train_full = train_df[candidate_features]
y_train = train_df['top10']
X_test_full = test_df[candidate_features]
y_test = test_df['top10']

print("Train shape:", X_train_full.shape, "Test shape:", X_test_full.shape)

# ============================================================
# 5. Forward selection: add features one at a time,
#    keep whichever addition improves accuracy the most
# ============================================================
def forward_selection(X, y, candidate_cols):
    selected = []
    remaining = list(candidate_cols)
    best_score_overall = 0
    history = []

    while remaining:
        scores = []
        for col in remaining:
            trial_cols = selected + [col]
            model = LogisticRegression(max_iter=1000)
            model.fit(X[trial_cols], y)
            score = model.score(X[trial_cols], y)   # training accuracy at this step
            scores.append((score, col))

        scores.sort(reverse=True)
        best_score, best_col = scores[0]

        if best_score > best_score_overall:
            best_score_overall = best_score
            selected.append(best_col)
            remaining.remove(best_col)
            history.append((best_col, best_score))
            print(f"Added '{best_col}' -> training accuracy: {best_score:.4f}")
        else:
            break   # no improvement, stop

    return selected, history

selected_features, selection_history = forward_selection(X_train_full, y_train, candidate_features)
print("\nFinal selected features:", selected_features)

# ============================================================
# 6. Train the final logistic regression model on selected features
# ============================================================
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_full[selected_features], y_train)

print("\nModel coefficients:")
for feat, coef in zip(selected_features, final_model.coef_[0]):
    print(f"  {feat}: {coef:.4f}")

Train shape: (1286, 10) Test shape: (356, 10)
Added 'grid' -> training accuracy: 0.7652
Added 'driver_recent_form' -> training accuracy: 0.7830
Added 'teammate_grid_position' -> training accuracy: 0.7869
Added 'track_temp' -> training accuracy: 0.7885

Final selected features: ['grid', 'driver_recent_form', 'teammate_grid_position', 'track_temp']

Model coefficients:
  grid: -0.1642
  driver_recent_form: -0.1412
  teammate_grid_position: -0.0556
  track_temp: -0.0011


In [63]:
# ============================================================
# 1. Predict on the held-out 2025 test set
# ============================================================
y_pred = final_model.predict(X_test_full[selected_features])
y_pred_proba = final_model.predict_proba(X_test_full[selected_features])[:, 1]

test_accuracy = accuracy_score(y_test, y_pred)
print(f"2025 Test Accuracy: {test_accuracy:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ============================================================
# 2. McFadden's Pseudo R^2 (the closest equivalent to R^2 for
#    logistic regression - NOT the same as regression R^2)
# ============================================================
from sklearn.linear_model import LogisticRegression

def mcfadden_pseudo_r2(model, X, y):
    # Log-likelihood of the fitted model
    proba = model.predict_proba(X)[:, 1]
    proba = np.clip(proba, 1e-10, 1 - 1e-10)  # avoid log(0)
    ll_model = np.sum(y * np.log(proba) + (1 - y) * np.log(1 - proba))

    # Log-likelihood of a "null" model (predicts the overall base rate for everyone)
    base_rate = y.mean()
    ll_null = np.sum(y * np.log(base_rate) + (1 - y) * np.log(1 - base_rate))

    return 1 - (ll_model / ll_null)

pseudo_r2_train = mcfadden_pseudo_r2(final_model, X_train_full[selected_features], y_train)
pseudo_r2_test = mcfadden_pseudo_r2(final_model, X_test_full[selected_features], y_test)

print(f"\nMcFadden's Pseudo R^2 (train): {pseudo_r2_train:.4f}")
print(f"McFadden's Pseudo R^2 (test):  {pseudo_r2_test:.4f}")

2025 Test Accuracy: 0.7584

Confusion Matrix:
[[126  33]
 [ 53 144]]

McFadden's Pseudo R^2 (train): 0.2875
McFadden's Pseudo R^2 (test):  0.2921


In [64]:
# ============================================================
# Build lookup tables from the most recent data, so the
# prediction function only needs driver name + grid + weather
# ============================================================

# Most recent known team + season points for each driver (from last available race)
latest_snapshot = df_all.sort_values('race_order_key').groupby('full_name').tail(1)
driver_team_lookup = latest_snapshot.set_index('full_name')['constructorName'].to_dict()
driver_team_points_lookup = latest_snapshot.set_index('full_name')['constructor_season_points_so_far'].to_dict()

# Driver's most recent form / dnf rate / experience / circuit history
driver_recent_form_lookup = latest_snapshot.set_index('full_name')['driver_recent_form'].to_dict()
driver_dnf_rate_lookup = latest_snapshot.set_index('full_name')['driver_dnf_rate_recent'].to_dict()
driver_experience_lookup = latest_snapshot.set_index('full_name')['driver_experience'].to_dict()

driver_circuit_lookup = (
    df_all.groupby(['full_name', 'race_name'])['position'].mean().to_dict()
)

# Fallback values for anything unknown (e.g. rookie drivers, new circuits)
fallback_grid_avg = df_all['grid'].mean()
fallback_position_avg = df_all['position'].mean()


def predict_top10(driver_name, circuit_name, grid, air_temp, track_temp, rainfall):
    """
    Predict whether a driver will finish in the Top 10.

    Parameters
    ----------
    driver_name : str   - e.g. 'Max Verstappen'
    circuit_name : str  - e.g. 'Monaco Grand Prix' (must match race_name format used in training)
    grid : int          - starting grid position
    air_temp : float    - expected air temperature (Celsius)
    track_temp : float  - expected track temperature (Celsius)
    rainfall : float    - 1 if rain expected, 0 if dry (fastf1 encodes this as boolean/float)
    """
    row = {
        'grid': grid,
        'constructor_season_points_so_far': driver_team_points_lookup.get(driver_name, 0),
        'driver_circuit_avg_position': driver_circuit_lookup.get(
            (driver_name, circuit_name), fallback_position_avg
        ),
        'driver_recent_form': driver_recent_form_lookup.get(driver_name, fallback_position_avg),
        'driver_dnf_rate_recent': driver_dnf_rate_lookup.get(driver_name, 0.1),
        # teammate_grid_position is unknown at prediction time (not provided as input) ->
        # use the average grid position as a neutral placeholder (see limitations below)
        'teammate_grid_position': fallback_grid_avg,
        'driver_experience': driver_experience_lookup.get(driver_name, 0),
        'air_temp': air_temp,
        'track_temp': track_temp,
        'rainfall': rainfall,
    }

    X_new = pd.DataFrame([row])[selected_features]
    proba = final_model.predict_proba(X_new)[0, 1]
    prediction = int(proba >= 0.5)

    return {
        'driver': driver_name,
        'circuit': circuit_name,
        'top10_prediction': bool(prediction),
        'top10_probability': round(proba, 3)
    }


# ============================================================
# Example usage for a 2026 race
# ============================================================
result = predict_top10(
    driver_name='Max Verstappen',
    circuit_name='Bahrain Grand Prix',
    grid=3,
    air_temp=28.0,
    track_temp=35.0,
    rainfall=0
)
print(result)

{'driver': 'Max Verstappen', 'circuit': 'Bahrain Grand Prix', 'top10_prediction': True, 'top10_probability': np.float64(0.888)}


# Task15 - [Advanced] Predictive Mini-Model (Optional Stretch)

# Nested Cross-Validation Analysis — Top10 Finish Prediction Model

This document describes a methodological correction made to the original
train/test evaluation of the Top10 finish prediction model, and the results
of a proper nested cross-validation using `TimeSeriesSplit`.

---

## 1. Background: The Problem With the Original Evaluation

The initial evaluation trained the model on 2020–2024 data, ran forward
selection **once** on that same 2020–2024 data to pick features, and then
tested the resulting model on the 2025 season.

This has a subtle but important flaw: **feature selection leakage**. When
feature selection and final evaluation share the same data, the reported
accuracy can be optimistically biased, because the features were chosen
with some knowledge of patterns present across the exact time range being
evaluated.

**Correction**: feature selection must be repeated independently *within
each* train/test split, using only that split's training data. This is
known as **nested cross-validation** — an "inner loop" (feature selection)
performed inside each "outer loop" (train/test fold).

---

## 2. Methodology

### 2.1 Splitting strategy: `TimeSeriesSplit`, grouped by race

Because the data is time-ordered (race results over multiple seasons),
a standard random k-fold split was avoided. `TimeSeriesSplit` was used
instead, which always trains on earlier data and tests on a later, unseen
block — mirroring how the model would actually be used to predict future
races.

Splitting was done at the **race level**, not the individual row level, so
that all drivers from the same race always remain together in the same
fold (avoiding any partial leakage of race-level information such as
weather or circuit conditions across the train/test boundary).

```python
tscv = TimeSeriesSplit(n_splits=5)
race_indices = np.arange(len(unique_races))
fold_boundaries = list(tscv.split(race_indices))
```

### 2.2 Candidate features (before selection)

```python
candidate_features = [
    'grid',
    'constructor_season_points_so_far',
    'driver_circuit_avg_position',
    'driver_recent_form',
    'driver_dnf_rate_recent',
    'teammate_grid_position',
    'driver_experience',
    'air_temp',
    'track_temp',
    'rainfall',
]
```

All features were engineered using `shift(1)` to ensure no feature for a
given race uses that race's own outcome (data leakage prevention).

### 2.3 Nested procedure (per fold)

For each of the 5 folds:
1. Forward selection was run **from scratch**, using only that fold's
   training rows, to select a feature subset.
2. A logistic regression model was trained on the training rows using
   only the selected features.
3. The model was evaluated on that fold's held-out test rows (a later,
   unseen block of races).

```python
for fold_num, (train_race_idx, test_race_idx) in enumerate(fold_boundaries):
    ...
    selected_this_fold = forward_selection(X_train_fold, y_train_fold, candidate_features)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_fold[selected_this_fold], y_train_fold)
    y_pred_fold = model.predict(X_test_fold[selected_this_fold])
```

This ensures each fold's reported accuracy reflects a fully independent
feature-selection-and-training process — an honest estimate of how the
whole pipeline (not just the final model) generalizes.

---

## 3. Results

### 3.1 Per-fold accuracy

| Fold | Train rows | Test rows | Accuracy |
|---|---|---|---|
| 1 | 298  | 276 | 0.7174 |
| 2 | 574  | 258 | 0.7907 |
| 3 | 832  | 302 | 0.8179 |
| 4 | 1134 | 268 | 0.7873 |
| 5 | 1402 | 240 | 0.7792 |

**Mean accuracy: 0.7785**
**Standard deviation: 0.0371**

### 3.2 Feature selection frequency across folds

| Feature | Folds selected in (out of 5) |
|---|---|
| `grid` | 5/5 |
| `track_temp` | 4/5 |
| `constructor_season_points_so_far` | 4/5 |
| `driver_recent_form` | 4/5 |
| `teammate_grid_position` | 3/5 |
| `rainfall` | 2/5 |
| `driver_circuit_avg_position` | 0/5 |
| `driver_dnf_rate_recent` | 0/5 |
| `driver_experience` | 0/5 |
| `air_temp` | 0/5 |

---

## 4. Interpretation

### 4.1 Model performance is stable, not a one-time fluke

Accuracy ranged from 71.7% to 81.8% across five independent, chronologically
separated folds, with a standard deviation of 3.71%. This range is
reasonably tight given the small per-fold sample sizes, and indicates the
pipeline's performance is not an artifact of any single lucky train/test
split.

### 4.2 `grid` is a universally strong predictor

`grid` (starting position) was selected in **every single fold**, regardless
of which time period it was trained on. This is the strongest possible
evidence available from this procedure that grid position carries genuine,
time-invariant predictive signal — consistent with the earlier finding of a
0.638 correlation between grid and finishing position.

### 4.3 Track temperature outperformed rainfall as a weather signal

`track_temp` was selected in 4 of 5 folds, while `rainfall` was selected in
only 2 of 5. This suggests track temperature (which affects tire grip and
degradation) may be a more consistent predictor of race unpredictability
than the simple presence/absence of rain — an interesting and
non-obvious finding worth highlighting.

### 4.4 Several features were never selected

`driver_circuit_avg_position`, `driver_dnf_rate_recent`, `driver_experience`,
and `air_temp` were not selected in any fold. This does not necessarily mean
these features carry no information in isolation — it may reflect that
their information overlaps substantially with features that were selected
first (e.g. `driver_recent_form` may already capture most of what
`driver_experience` would add), and forward selection tends to stop once
marginal gains become small.

### 4.5 Increased instability compared to the earlier (uncorrected) experiment

The standard deviation here (3.71%) is higher than in the earlier,
methodologically flawed experiment (2.71%), where a single, fixed feature
set (chosen using the full 2020–2024 range) was applied uniformly across
all folds. This increase is expected and is itself informative: it shows
that allowing feature selection to vary per fold — as it honestly should —
introduces additional variance that the earlier setup had artificially
suppressed. In other words, the earlier estimate was likely **modestly
optimistic**, and this nested result is the more defensible one.

### 4.6 Fold 1's comparatively low accuracy (71.7%)

Fold 1 had by far the smallest training set (298 rows), which may partly
explain its lower accuracy. It's also possible this early period
(2020) contained a higher share of atypical race outcomes. Distinguishing
between "small sample size" and "genuinely harder period to predict" would
require further investigation (e.g. examining accident/DNF rates specific
to that period).

---

## 5. Implications for the Final Production Model

Nested cross-validation is designed to produce an **honest performance
estimate**, not to select the final feature set for deployment. Two
reasonable approaches for finalizing the production model:

1. **Re-run forward selection once on the full 2020–2025 dataset.** This
   gives the model access to the maximum amount of data when making the
   final feature choice.
2. **Restrict to "stable" features** — those selected in a majority of
   folds (e.g. ≥3/5): `grid`, `track_temp`, `constructor_season_points_so_far`,
   `driver_recent_form`, `teammate_grid_position`. This is a more
   conservative choice that excludes potentially fold-specific noise.

Reporting both approaches side by side, and noting where they agree, is the
strongest way to justify the final feature set.

---

## 6. Summary

Nested cross-validation with `TimeSeriesSplit` corrected a feature selection
leakage issue in the original evaluation. The corrected estimate shows a
mean accuracy of 77.85% (SD 3.71%) across five chronologically ordered
folds. Grid position emerged as a universally selected, time-invariant
predictor, while several engineered features (driver circuit history, DNF
rate, experience, air temperature) were not selected in any fold and may
warrant reconsideration or removal from the final feature set.

In [75]:
# Step 1: Imports and chronological sorting.
# TimeSeriesSplit requires the data to be in time order before splitting.
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, confusion_matrix

df_sorted = df_all.sort_values('race_order_key').reset_index(drop=True)

print("Year range:", df_sorted['year'].min(), "-", df_sorted['year'].max())
print("Total rows:", len(df_sorted))

Year range: 2020 - 2025
Total rows: 2618


In [76]:
# Step 2: Define candidate features (before any selection) and drop rows
# with missing values in these columns. Missing values happen for a
# driver's/team's earliest races, where there's no history yet to
# compute rolling/expanding features.
candidate_features = [
    'grid',
    'constructor_season_points_so_far',
    'driver_circuit_avg_position',
    'driver_recent_form',
    'driver_dnf_rate_recent',
    'teammate_grid_position',
    'driver_experience',
    'air_temp',
    'track_temp',
    'rainfall',
]

print("Rows before dropping NaN:", len(df_sorted))
df_sorted = df_sorted.dropna(subset=candidate_features + ['top10']).reset_index(drop=True)
print("Rows after dropping NaN:", len(df_sorted))

Rows before dropping NaN: 2618
Rows after dropping NaN: 1642


In [77]:
# Step 3: Get unique races in chronological order.
# We split by RACE (not by individual row) so all drivers from the
# same race always stay together in the same fold - this avoids
# leaking information about a race's weather/circuit across the split.
unique_races = df_sorted['race_order_key'].drop_duplicates().sort_values().values
print("Number of unique races:", len(unique_races))

Number of unique races: 98


In [78]:
# Step 4: Create fold boundaries using TimeSeriesSplit, applied to the
# race index (not the row index). Each fold trains on earlier races and
# tests on a later, unseen block of races.
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

race_indices = np.arange(len(unique_races))
fold_boundaries = list(tscv.split(race_indices))

print(f"Created {len(fold_boundaries)} folds")
for i, (train_race_idx, test_race_idx) in enumerate(fold_boundaries):
    print(f"Fold {i+1}: train races = {len(train_race_idx)}, test races = {len(test_race_idx)}")

Created 5 folds
Fold 1: train races = 18, test races = 16
Fold 2: train races = 34, test races = 16
Fold 3: train races = 50, test races = 16
Fold 4: train races = 66, test races = 16
Fold 5: train races = 82, test races = 16


In [79]:
# Step 5: Forward selection function.
# This will be called FRESH inside each fold, using only that fold's
# training data - never the full dataset - to avoid feature selection leakage.
def forward_selection(X, y, candidate_cols):
    selected = []
    remaining = list(candidate_cols)
    best_score_overall = 0

    while remaining:
        scores = []
        for col in remaining:
            trial_cols = selected + [col]
            model = LogisticRegression(max_iter=1000)
            model.fit(X[trial_cols], y)
            score = model.score(X[trial_cols], y)
            scores.append((score, col))

        scores.sort(reverse=True)
        best_score, best_col = scores[0]

        if best_score > best_score_overall:
            best_score_overall = best_score
            selected.append(best_col)
            remaining.remove(best_col)
        else:
            break

    return selected

In [80]:
# Step 6: Nested cross-validation loop.
# For EACH fold: run forward selection using only that fold's training
# data, train a logistic regression on the selected features, then
# evaluate on that fold's held-out test races.
fold_results = []

for fold_num, (train_race_idx, test_race_idx) in enumerate(fold_boundaries):
    train_races = unique_races[train_race_idx]
    test_races = unique_races[test_race_idx]

    train_mask = df_sorted['race_order_key'].isin(train_races)
    test_mask = df_sorted['race_order_key'].isin(test_races)

    X_train_fold = df_sorted.loc[train_mask, candidate_features]
    y_train_fold = df_sorted.loc[train_mask, 'top10']
    X_test_fold = df_sorted.loc[test_mask, candidate_features]
    y_test_fold = df_sorted.loc[test_mask, 'top10']

    # Forward selection re-run from scratch, using ONLY this fold's training data
    selected_this_fold = forward_selection(X_train_fold, y_train_fold, candidate_features)

    # Train final model for this fold on the selected features only
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_fold[selected_this_fold], y_train_fold)

    y_pred_fold = model.predict(X_test_fold[selected_this_fold])
    acc = accuracy_score(y_test_fold, y_pred_fold)

    fold_results.append({
        'fold': fold_num + 1,
        'train_rows': len(X_train_fold),
        'test_rows': len(X_test_fold),
        'accuracy': acc,
        'selected_features': selected_this_fold
    })

    print(f"Fold {fold_num+1}: accuracy={acc:.4f}")
    print(f"  Selected features: {selected_this_fold}")

Fold 1: accuracy=0.7174
  Selected features: ['grid', 'track_temp']
Fold 2: accuracy=0.7907
  Selected features: ['grid', 'constructor_season_points_so_far', 'driver_recent_form', 'teammate_grid_position']
Fold 3: accuracy=0.8179
  Selected features: ['grid', 'driver_recent_form', 'constructor_season_points_so_far', 'rainfall', 'track_temp']
Fold 4: accuracy=0.7873
  Selected features: ['grid', 'constructor_season_points_so_far', 'driver_recent_form', 'track_temp', 'rainfall', 'teammate_grid_position']
Fold 5: accuracy=0.7792
  Selected features: ['grid', 'constructor_season_points_so_far', 'driver_recent_form', 'track_temp', 'teammate_grid_position']


In [81]:
# Step 7: Summarize accuracy across all folds
results_df = pd.DataFrame(fold_results)
print(results_df[['fold', 'train_rows', 'test_rows', 'accuracy']])

print(f"\nMean accuracy across {n_splits} folds: {results_df['accuracy'].mean():.4f}")
print(f"Std deviation of accuracy: {results_df['accuracy'].std():.4f}")

   fold  train_rows  test_rows  accuracy
0     1         298        276  0.717391
1     2         574        258  0.790698
2     3         832        302  0.817881
3     4        1134        268  0.787313
4     5        1402        240  0.779167

Mean accuracy across 5 folds: 0.7785
Std deviation of accuracy: 0.0371


In [82]:
# Step 8: Check how consistent the selected features are across folds.
# If the same features keep getting picked, that's strong evidence
# they are genuinely stable predictors (not just a fluke of one split).
from collections import Counter

all_selected = [feat for row in fold_results for feat in row['selected_features']]
feature_counts = Counter(all_selected)

print("How many folds (out of", n_splits, ") each feature was selected in:")
for feat, count in feature_counts.most_common():
    print(f"  {feat}: {count}/{n_splits}")

How many folds (out of 5 ) each feature was selected in:
  grid: 5/5
  track_temp: 4/5
  constructor_season_points_so_far: 4/5
  driver_recent_form: 4/5
  teammate_grid_position: 3/5
  rainfall: 2/5


In [83]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression

# Use the features that were selected in at least 3 out of 5 folds
# (from the nested cross-validation analysis)
final_features = [
    'grid',
    'track_temp',
    'constructor_season_points_so_far',
    'driver_recent_form',
    'teammate_grid_position',
]

# Train on ALL available data (2020-2025) for the production model
X_final = df_sorted[final_features]
y_final = df_sorted['top10']

final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_final, y_final)

print("Model coefficients:")
for feat, coef in zip(final_features, final_model.coef_[0]):
    print(f"  {feat}: {coef:.4f}")

Model coefficients:
  grid: -0.1732
  track_temp: -0.0030
  constructor_season_points_so_far: 0.0030
  driver_recent_form: -0.0921
  teammate_grid_position: -0.0289


In [84]:
# Get each driver's most recent known values (last race they appeared in)
latest_snapshot = df_sorted.sort_values('race_order_key').groupby('full_name').tail(1)

driver_recent_form_lookup = latest_snapshot.set_index('full_name')['driver_recent_form'].to_dict()
constructor_points_lookup = latest_snapshot.set_index('full_name')['constructor_season_points_so_far'].to_dict()

fallback_form = df_sorted['driver_recent_form'].mean()
fallback_points = df_sorted['constructor_season_points_so_far'].mean()

In [85]:
def predict_top10(driver_name, grid, track_temp, teammate_grid):
    """
    Predict whether a driver finishes in the Top 10.

    Parameters
    ----------
    driver_name : str   - e.g. 'Max Verstappen'
    grid : int           - this race's starting grid position
    track_temp : float   - expected track temperature (Celsius)
    teammate_grid : int  - teammate's starting grid position this race
    """
    row = {
        'grid': grid,
        'track_temp': track_temp,
        'constructor_season_points_so_far': constructor_points_lookup.get(driver_name, fallback_points),
        'driver_recent_form': driver_recent_form_lookup.get(driver_name, fallback_form),
        'teammate_grid_position': teammate_grid,
    }

    X_new = pd.DataFrame([row])[final_features]
    proba = final_model.predict_proba(X_new)[0, 1]
    prediction = int(proba >= 0.5)

    return {
        'driver': driver_name,
        'grid': grid,
        'top10_prediction': bool(prediction),
        'top10_probability': round(proba, 3)
    }

In [93]:
# Example: predicting a 2026 race
result = predict_top10(
    driver_name='Max Verstappen',
    grid=1,
    track_temp=34.5,
    teammate_grid=6
)
print(result)

{'driver': 'Max Verstappen', 'grid': 1, 'top10_prediction': True, 'top10_probability': np.float64(0.961)}


# Model Evaluation Summary — Two Versions

## Version 1: Original Evaluation (Single Split)

Trained on 2020–2024, tested once on the full 2025 season.

| Metric | Value |
|---|---|
| Accuracy | 75.84% |
| Pseudo R² (train) | 0.2875 |
| Pseudo R² (test) | 0.2921 |
| Precision | 81.4% |
| Recall | 73.1% |

**Limitation**: features were selected using the same 2020–2024 range later used for the final model — a form of feature selection leakage that can optimistically bias results.

---

## Version 2: Corrected Evaluation (Nested Cross-Validation)

Used `TimeSeriesSplit` (5 folds, race-level grouping), re-running forward selection independently inside each fold.

| Fold | Train rows | Test rows | Accuracy |
|---|---|---|---|
| 1 | 298  | 276 | 71.7% |
| 2 | 574  | 258 | 79.1% |
| 3 | 832  | 302 | 81.8% |
| 4 | 1134 | 268 | 78.7% |
| 5 | 1402 | 240 | 77.9% |
| **Mean** | — | — | **77.85% (SD 3.71%)** |

| Feature | Folds selected in (/5) |
|---|---|
| `grid` | 5/5 |
| `track_temp` | 4/5 |
| `constructor_season_points_so_far` | 4/5 |
| `driver_recent_form` | 4/5 |
| `teammate_grid_position` | 3/5 |
| `rainfall` | 2/5 |

---

## Key Insights

1. **`grid` is the single most reliable predictor** — selected in every fold, regardless of time period, consistent with its 0.638 correlation with finishing position found earlier.
2. **Track temperature matters more than rain itself** — `track_temp` (4/5) outperformed `rainfall` (2/5) as a weather signal, suggesting tire/grip effects are more predictive than simple wet/dry conditions.
3. **Higher variance after correction is a feature, not a bug** — SD rose from 2.71% (flawed setup) to 3.71% (corrected), indicating the original single-split result was modestly optimistic.
4. **Some engineered features were never selected** (`driver_circuit_avg_position`, `driver_dnf_rate_recent`, `driver_experience`, `air_temp`), likely due to overlapping information with stronger features rather than having zero predictive value on their own.

## Final Decision
Production model uses features selected in ≥3/5 folds: `grid`, `track_temp`, `constructor_season_points_so_far`, `driver_recent_form`, `teammate_grid_position`.